# Métodologia de recorte

Usaremos uma função capaz de recortar patches com sobreposição, dada em pixels, de uma região de interesse (ROI) nas imagens. 
Patches resultantes não podem ter tamnho diferente do especificado e o georreferenciamento é preservado.

In [1]:
import os
from osgeo import gdal

def recortar_patches(
    image_path,
    patch_rows,
    patch_cols,
    out_dir,
    roi,
    roi_id = "",
    overlap=(0, 0),
    compress_lzw=True
):
    """
    Funcão para recorte de patches de uma imagem usando GDAL.

    Parameters
    ----------
    image_path : str
        Caminho para a imagem de entrada.
    patch_rows : int
        Número de linhas (altura) de cada patch em pixels.
    patch_cols : int
        Número de colunas (largura) de cada patch em pixels.
    out_dir : str
        Diretório de saída.
    roi : tuple
        Região de interesse (x, y, w, h) em coordenadas de pixel.
    roi_id : str
        Identificador da região de interesse usado no nome dos arquivos gravados.
    overlap : tuple, optional
        Sobreposição (overlap_y, overlap_x) em pixels.
    compress_lzw : bool, optional
        Se True, grava os patches com compressão LZW.
    """

    ds = gdal.Open(image_path, gdal.GA_ReadOnly)
    if ds is None:
        raise RuntimeError(f"Não foi possível abrir a imagem: {image_path}")

    base_name = os.path.splitext(os.path.basename(image_path))[0]
    os.makedirs(out_dir, exist_ok=True)

    x0, y0, w, h = roi
    step_y = patch_rows - overlap[0]
    step_x = patch_cols - overlap[1]

    if step_x <= 0 or step_y <= 0:
        raise ValueError("Overlap maior ou igual ao tamanho do patch.")

    gt = ds.GetGeoTransform()
    proj = ds.GetProjection()
    bands = ds.RasterCount
    dtype = ds.GetRasterBand(1).DataType

    driver = gdal.GetDriverByName("GTiff")

    creation_options = []
    if compress_lzw:
        creation_options.append("COMPRESS=LZW")

    i = 0
    for y in range(y0, y0 + h - patch_rows + 1, step_y):
        j = 0
        for x in range(x0, x0 + w - patch_cols + 1, step_x):

            patch_data = ds.ReadAsArray(x, y, patch_cols, patch_rows)
            if patch_data is None:
                j += 1
                continue

            out_name = (
                f"{base_name}_{roi_id}_{i}_{j}.tif"
            )
            out_path = os.path.join(out_dir, out_name)

            out_ds = driver.Create(
                out_path,
                patch_cols,
                patch_rows,
                bands,
                dtype,
                options=creation_options
            )

            # Ajuste da geotransformação
            new_gt = list(gt)
            new_gt[0] = gt[0] + x * gt[1] + y * gt[2]
            new_gt[3] = gt[3] + x * gt[4] + y * gt[5]
            out_ds.SetGeoTransform(tuple(new_gt))
            out_ds.SetProjection(proj)

            if bands == 1:
                out_ds.GetRasterBand(1).WriteArray(patch_data)
            else:
                for b in range(bands):
                    out_ds.GetRasterBand(b + 1).WriteArray(patch_data[b])

            out_ds.FlushCache()
            out_ds = None

            j += 1
        i += 1

    ds = None


Com isto é possivel subdividir os mosaicos de grandes dimensões usando diferentes ROI. Assumimos que cada imagem pode ser dividida em um grid com 2x3 ROIs. Adotando 4 imagens temos 24 ROIs. São as imagens em questão (no caminho relativo ../data/rio/6_band_data_2024_1_5000/):

- RJ286D2_6band.tif
- RJ287A4_6band.tif
- RJ287C1_6band.tif
- RJ287C2_6band.tif

Seus respectivos tamanhos são:

In [2]:
!gdalinfo ../data/rio/6_band_data_2024_1_5000/RJ286D2_6band.tif | grep Size\ is
!gdalinfo ../data/rio/6_band_data_2024_1_5000/RJ287A4_6band.tif | grep Size\ is
!gdalinfo ../data/rio/6_band_data_2024_1_5000/RJ287C1_6band.tif | grep Size\ is
!gdalinfo ../data/rio/6_band_data_2024_1_5000/RJ287C2_6band.tif | grep Size\ is

Size is 21818, 15899
Size is 21828, 15909
Size is 21822, 15904
Size is 21824, 15909


Assumindo duas linhas e três colunas por imagem e usando os multiplos inteiros que não superam os menores valores observados no conjunto temos:

In [3]:
ROI_size = 15899//2, 21818//3
ROI_size

(7949, 7272)

In [4]:
2*ROI_size[0], 3*ROI_size[1] 

(15898, 21816)

Assumindo uma sobreposicão de 1/12 (8,3%):

In [5]:
patche_size = 384
overlap = patche_size//12
overlap

32

O loop de recorte então é:

In [ ]:
r_path = "../data/rio/6_band_data_2024_1_5000/"
images = ["RJ286D2_6band.tif",
          "RJ287A4_6band.tif",
          "RJ287C1_6band.tif",
          "RJ287C2_6band.tif"]

for i, image in enumerate(images):
    for m in range(2):
        for n in range(3):
            recortar_patches(image_path=r_path+image,
                             patch_rows=patche_size,patch_cols=patche_size,
                             out_dir="../data/rio/building_patches/all_images",
                             roi=(n*ROI_size[1], m*ROI_size[0], ROI_size[1], ROI_size[0]),
                             roi_id=f"{m}_{n}",
                             overlap=(overlap, overlap))

O mesmo precisa ser feito para os rótulos:

In [7]:
!gdalinfo ../data/rio/render_masks_2024_1_5000/RJ286D2_mask.tif | grep Size\ is
!gdalinfo ../data/rio/render_masks_2024_1_5000/RJ287A4_mask.tif | grep Size\ is
!gdalinfo ../data/rio/render_masks_2024_1_5000/RJ287C1_mask.tif | grep Size\ is
!gdalinfo ../data/rio/render_masks_2024_1_5000/RJ287C2_mask.tif | grep Size\ is

Size is 21818, 15899
Size is 21828, 15909
Size is 21822, 15904
Size is 21824, 15909


In [8]:
r_path = "../data/rio/render_masks_2024_1_5000/"
images = ["RJ286D2_mask.tif",
          "RJ287A4_mask.tif",
          "RJ287C1_mask.tif",
          "RJ287C2_mask.tif"]

for i, image in enumerate(images):
    for m in range(2):
        for n in range(3):
            recortar_patches(image_path=r_path+image,
                             patch_rows=patche_size,patch_cols=patche_size,
                             out_dir="../data/rio/building_patches/all_labels",
                             roi=(n*ROI_size[1], m*ROI_size[0], ROI_size[1], ROI_size[0]),
                             roi_id=f"{m}_{n}",
                             overlap=(overlap, overlap))

É necessário ajustar o nome dos arquivos gerados para garantir que eles iram corresponder com facilidade:

In [9]:
def renomear(directory_path, text_to_remove):
    for filename in os.listdir(directory_path):
        # Constrói o caminho completo para o arquivo antigo
        old_file_path = os.path.join(directory_path, filename)

        # Verifica se é um arquivo (e não um subdiretório)
        if os.path.isfile(old_file_path):
            # Cria o novo nome do arquivo, removendo o texto indesejado
            new_filename = filename.replace(text_to_remove, "")
            # Constrói o caminho completo para o novo arquivo
            new_file_path = os.path.join(directory_path, new_filename)

            # Renomeia o arquivo
            try:
                os.rename(old_file_path, new_file_path)
                print(f"Renomeado: '{filename}' para '{new_filename}'")
            except OSError as e:
                print(f"Erro ao renomear {filename}: {e}")

renomear("../data/rio/building_patches/all_images","_6band")
renomear("../data/rio/building_patches/all_labels","_mask")

Renomeado: 'RJ286D2_6band_1_0_3_18.tif' para 'RJ286D2_1_0_3_18.tif'
Renomeado: 'RJ286D2_6band_0_1_13_10.tif' para 'RJ286D2_0_1_13_10.tif'
Renomeado: 'RJ287C2_6band_1_1_10_8.tif' para 'RJ287C2_1_1_10_8.tif'
Renomeado: 'RJ287C2_6band_1_1_13_1.tif' para 'RJ287C2_1_1_13_1.tif'
Renomeado: 'RJ287C1_6band_1_0_4_17.tif' para 'RJ287C1_1_0_4_17.tif'
Renomeado: 'RJ287C2_6band_1_1_2_15.tif' para 'RJ287C2_1_1_2_15.tif'
Renomeado: 'RJ287C1_6band_0_1_8_3.tif' para 'RJ287C1_0_1_8_3.tif'
Renomeado: 'RJ287C2_6band_0_1_9_7.tif' para 'RJ287C2_0_1_9_7.tif'
Renomeado: 'RJ287C1_6band_0_1_19_11.tif' para 'RJ287C1_0_1_19_11.tif'
Renomeado: 'RJ287C1_6band_0_1_0_11.tif' para 'RJ287C1_0_1_0_11.tif'
Renomeado: 'RJ287A4_6band_0_2_20_8.tif' para 'RJ287A4_0_2_20_8.tif'
Renomeado: 'RJ287C2_6band_0_1_3_3.tif' para 'RJ287C2_0_1_3_3.tif'
Renomeado: 'RJ286D2_6band_1_1_8_9.tif' para 'RJ286D2_1_1_8_9.tif'
Renomeado: 'RJ286D2_6band_0_0_17_12.tif' para 'RJ286D2_0_0_17_12.tif'
Renomeado: 'RJ287C2_6band_0_0_18_6.tif' para 'RJ28

Para reduzir o número de imagens que precisam ser analisadas e melhorar o balanceamento de classes vamos eliminar regiões que não tenham menos de 1% dos pixels marcados como edificacão.

In [10]:
import os
from osgeo import gdal
import numpy as np


def remover_imagens_por_limiar(
    input_dir,
    pixel_value,
    min_count,
    ref_dir=None,
    band_index=1,
    extensions=(".tif", ".tiff")
):
    """
    Remove imagens que não possuem quantidade suficiente de pixels
    com um determinado valor.

    Parameters
    ----------
    input_dir : str
        Diretório contendo as imagens a serem avaliadas.
    pixel_value : int or float
        Valor do pixel a ser contado.
    min_count : int
        Limiar mínimo de pixels com o valor especificado.
    ref_dir : str, optional
        Diretório de referência com imagens de mesmo nome.
        Se fornecido, as imagens serão removidas em ambos os diretórios.
    band_index : int, optional
        Banda a ser avaliada (1-based). Default = 1.
    extensions : tuple, optional
        Extensões de arquivos consideradas.
    """

    input_files = [
        f for f in os.listdir(input_dir)
        if f.lower().endswith(extensions)
    ]

    for fname in input_files:
        input_path = os.path.join(input_dir, fname)

        ds = gdal.Open(input_path, gdal.GA_ReadOnly)
        if ds is None:
            print(f"[WARN] Não foi possível abrir {input_path}")
            continue

        if band_index > ds.RasterCount:
            ds = None
            raise ValueError(
                f"Banda {band_index} não existe em {fname}"
            )

        band = ds.GetRasterBand(band_index)
        arr = band.ReadAsArray()
        ds = None

        if arr is None:
            print(f"[WARN] Falha ao ler dados de {fname}")
            continue

        count = np.count_nonzero(arr == pixel_value)

        if count < min_count:
            print(
                f"[REMOVE] {fname} | pixels={count} < limiar={min_count}"
            )

            # Remove do diretório principal
            os.remove(input_path)

            # Remove do diretório de referência, se existir
            if ref_dir is not None:
                ref_path = os.path.join(ref_dir, fname)
                if os.path.exists(ref_path):
                    os.remove(ref_path)


In [11]:
limiar = patche_size*patche_size//100
limiar

1474

In [12]:
remover_imagens_por_limiar(
    input_dir="../data/rio/building_patches/all_labels",
    pixel_value=255,
    min_count=limiar,
    ref_dir="../data/rio/building_patches/all_images",
    band_index=1
)

[REMOVE] RJ286D2_1_0_21_0.tif | pixels=0 < limiar=1474
[REMOVE] RJ287C1_1_2_20_3.tif | pixels=0 < limiar=1474
[REMOVE] RJ287C1_1_2_10_19.tif | pixels=0 < limiar=1474
[REMOVE] RJ286D2_1_1_10_16.tif | pixels=0 < limiar=1474
[REMOVE] RJ286D2_1_2_19_1.tif | pixels=61 < limiar=1474
[REMOVE] RJ286D2_0_0_4_4.tif | pixels=0 < limiar=1474
[REMOVE] RJ286D2_1_0_5_7.tif | pixels=0 < limiar=1474
[REMOVE] RJ286D2_1_1_11_10.tif | pixels=0 < limiar=1474
[REMOVE] RJ286D2_0_1_1_9.tif | pixels=0 < limiar=1474
[REMOVE] RJ286D2_1_0_11_16.tif | pixels=0 < limiar=1474
[REMOVE] RJ286D2_1_1_19_8.tif | pixels=0 < limiar=1474
[REMOVE] RJ287C1_1_2_11_16.tif | pixels=0 < limiar=1474
[REMOVE] RJ287A4_1_1_6_5.tif | pixels=0 < limiar=1474
[REMOVE] RJ286D2_1_1_16_6.tif | pixels=0 < limiar=1474
[REMOVE] RJ286D2_1_0_6_6.tif | pixels=0 < limiar=1474
[REMOVE] RJ286D2_0_0_10_3.tif | pixels=0 < limiar=1474
[REMOVE] RJ287C2_0_1_13_17.tif | pixels=0 < limiar=1474
[REMOVE] RJ286D2_0_0_20_18.tif | pixels=0 < limiar=1474
[REMOVE

Contabilizando o total de patches e espaço usado:

In [ ]:
!ls ../data/rio/building_patches/all/labels | wc -l
!ls ../data/rio/building_patches/all/images | wc -l
!du -hs ../data/rio/building_patches/all

Identificando o total de patches por imagem e região de interesse:

In [14]:
import glob

preffix_imgs = {}
for image in glob.glob("../data/rio/building_patches/all_labels/*.tif"):
    pattern = os.path.basename(image).split("_")
    if len(pattern) > 3:
        name = "_".join(pattern[:3])
        if name in preffix_imgs:
            preffix_imgs[name] += 1
        else:
            preffix_imgs[name] = 1

for i, key in enumerate(sorted(preffix_imgs.keys())):
    print(i+1, key, preffix_imgs[key])

total_imgs = sum(preffix_imgs.values())
total_imgs

1 RJ286D2_0_0 35
2 RJ286D2_0_1 337
3 RJ286D2_0_2 428
4 RJ286D2_1_1 170
5 RJ286D2_1_2 380
6 RJ287A4_0_0 379
7 RJ287A4_0_1 409
8 RJ287A4_0_2 384
9 RJ287A4_1_0 366
10 RJ287A4_1_1 378
11 RJ287A4_1_2 392
12 RJ287C1_0_0 436
13 RJ287C1_0_1 438
14 RJ287C1_0_2 428
15 RJ287C1_1_0 417
16 RJ287C1_1_1 351
17 RJ287C1_1_2 147
18 RJ287C2_0_0 436
19 RJ287C2_0_1 419
20 RJ287C2_0_2 412
21 RJ287C2_1_0 203
22 RJ287C2_1_1 344
23 RJ287C2_1_2 346


8035

Vamos escolher quais regiões de interesse serão destinados a teste e validação usando uma soma aproximada de 1/5 (20%):

In [15]:
def subset_sum_aprox_indices(valores, alvo, tol=None):
    """
    Retorna os índices do subconjunto de 'valores' cuja soma
    é a mais próxima possível de 'alvo'.

    Parâmetros
    ----------
    valores : list[float]
        Lista de valores numéricos
    alvo : float
        Valor alvo da soma
    tol : float ou None
        Tolerância para poda (quanto maior, mais rápido e menos preciso).
        Se None, nenhuma poda é aplicada.

    Retorno
    -------
    list[int]
        Índices do subconjunto escolhido
    """

    # dicionário: soma -> lista de índices
    estados = {0.0: []}

    for i, v in enumerate(valores):
        novos_estados = {}

        for soma_atual, idxs in estados.items():
            nova_soma = soma_atual + v
            novos_estados[nova_soma] = idxs + [i]

        estados.update(novos_estados)

        # poda para evitar explosão combinatória
        if tol is not None:
            estados = {
                s: idxs
                for s, idxs in estados.items()
                if abs(s - alvo) <= tol
            }

    # escolhe a soma mais próxima do alvo
    melhor_soma = min(estados.keys(), key=lambda s: abs(s - alvo))
    return estados[melhor_soma]

test_and_val_idxs = subset_sum_aprox_indices(preffix_imgs.values(), total_imgs//5)

In [16]:
all_keys = list(preffix_imgs.keys())
all_values = list(preffix_imgs.values())
test_and_val_imgs = [all_keys[i] for i in test_and_val_idxs]
test_and_val_values = [all_values[i] for i in test_and_val_idxs] 
test_and_val_sum = sum(test_and_val_values)

O conjunto de imagens de teste e validação deve ainda ser subdividido:

In [17]:
# Distribuição do conjunto de dados anterior usada como referência
479/(479+1436+7661),1436/(479+1436+7661),7661/(479+1436+7661)

(0.050020885547201334, 0.14995822890559732, 0.8000208855472013)

In [18]:
subset_test_and_val = dict(zip(test_and_val_imgs, test_and_val_values))
test_idxs = subset_sum_aprox_indices(subset_test_and_val.values(), test_and_val_sum//2)

test_and_val_keys = list(subset_test_and_val.keys())
test_and_val_values = list(subset_test_and_val.values())
val_imgs = [test_and_val_keys[i] for i in test_idxs]
val_values = [test_and_val_values[i] for i in test_idxs] 
total_val_imgs = sum(val_values)
print(sorted(val_imgs), total_val_imgs, test_and_val_sum//2)

['RJ286D2_0_0', 'RJ287C1_1_1', 'RJ287C2_0_2'] 798 803


In [19]:
test_imgs = [img for img in test_and_val_imgs if img not in val_imgs]
train_imgs = [img for img in all_keys if img not in test_and_val_imgs]

In [20]:
dataset = {"test":test_imgs, "val":val_imgs, "train":train_imgs}
dataset

{'test': ['RJ287C2_1_0', 'RJ287C2_0_0', 'RJ286D2_1_1'],
 'val': ['RJ287C1_1_1', 'RJ287C2_0_2', 'RJ286D2_0_0'],
 'train': ['RJ286D2_1_2',
  'RJ287C1_0_0',
  'RJ287A4_1_0',
  'RJ287C1_1_0',
  'RJ287A4_0_1',
  'RJ287A4_0_2',
  'RJ287A4_0_0',
  'RJ287C1_0_2',
  'RJ287A4_1_1',
  'RJ286D2_0_1',
  'RJ287A4_1_2',
  'RJ286D2_0_2',
  'RJ287C1_0_1',
  'RJ287C2_0_1',
  'RJ287C2_1_2',
  'RJ287C2_1_1',
  'RJ287C1_1_2']}

In [21]:
dataset_size = {}
for subset in dataset:
    dataset_size[subset] = 0
    for img in dataset[subset]:
        dataset_size[subset] += preffix_imgs[img]
dataset_size

{'test': 809, 'val': 798, 'train': 6428}

In [23]:
import os
import shutil

def move_dataset_files(dir_origem, regras, sufixo=None):
    """
    Move arquivos de um diretório de origem para subdiretórios irmãos,
    com base em substrings presentes no nome dos arquivos.

    Parameters
    ----------
    dir_origem : str
        Caminho do diretório de origem (ex: /home/eduardo)
    regras : dict
        Dicionário onde:
          - chave   -> nome do diretório de destino (ex: 'alberto')
          - valor   -> lista de substrings que identificam os arquivos
    sufixo : str or None
        Subdiretório adicional a ser criado dentro do destino
        (ex: 'dados' → /home/alberto/dados)
    """

    dir_origem = os.path.abspath(dir_origem)
    dir_pai = os.path.dirname(dir_origem)

    for nome_arquivo in os.listdir(dir_origem):
        caminho_arquivo = os.path.join(dir_origem, nome_arquivo)

        # ignora subdiretórios
        if not os.path.isfile(caminho_arquivo):
            continue

        for dir_destino_nome, padroes in regras.items():
            if any(padrao in nome_arquivo for padrao in padroes):

                # monta caminho do destino
                if sufixo:
                    dir_destino = os.path.join(dir_pai, dir_destino_nome, sufixo)
                else:
                    dir_destino = os.path.join(dir_pai, dir_destino_nome)

                os.makedirs(dir_destino, exist_ok=True)

                destino_final = os.path.join(dir_destino, nome_arquivo)
                shutil.move(caminho_arquivo, destino_final)

                # evita múltiplos matches para o mesmo arquivo
                break

move_dataset_files("../data/rio/building_patches/all_labels", dataset, sufixo="labels")
move_dataset_files("../data/rio/building_patches/all_images", dataset, sufixo="images")

**Observacão:** O conjunto de dados recortado compreende imagens aéreas de 2024 e rótulos para edificacões de 2019. Logo, apesar de tratar-se de uma região mais consolidada do município, não é certo assumir que todos os rótulos estão necessáriamente corretos. 
A decisão adotada neste projeto foi por experimentar o treinamento do conjunto de dados sem curadoria e confrontar os resultados com outro treinamento após a curadoria. No processo de curadoria adotado optou-se pela selecão e posterior remocão de quais recortes de imagens onde a verificacão visual identificasse acréscimos ou ausencias de edificacões ou de suas partes significativas.